In [24]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent



In [25]:
candidate_file = (
    project_root
    / "data"
    / "processed"
    / "atm_candidates_with_variant_id.csv"
)

candidate_variants = pd.read_csv(candidate_file)
print("Candidates loaded:",len(candidate_variants))

candidate_variants.head()

Candidates loaded: 4534


,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,Oncogenicity date last evaluated,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,variant_id
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,NaN,11:108098355,11:108227628,4170131,VCV004170131,rs4279232,11-108227628-A-C
1,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,NaN,11:108098355,11:108227628,640846,VCV000640846,rs639367,11-108227628-A-G
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,NaN,11:108098356,11:108227629,181943,VCV000181943,rs180377,11-108227629-G-C
3,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,NaN,11:108098357,11:108227630,1337452,VCV001337452,rs1328461,11-108227630-T-G
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,NaN,11:108098357,11:108227630,922075,VCV000922075,rs911284,11-108227630-T-A


In [26]:
import requests

gnomad_url = "https://gnomad.broadinstitute.org/api"

query = """
{
  gene(gene_symbol: "ATM", reference_genome: GRCh38) {
    variants(dataset: gnomad_r4) {
      variant_id
      joint {
        ac
        an
      }
    }
  }
}
"""

response = requests.post(
    gnomad_url,
    json={"query": query},
    timeout=120
)

response.raise_for_status()

gnomad_response = response.json()

if "errors" in gnomad_response:
    print(gnomad_response["errors"])
else:
    gnomad_variants = gnomad_response["data"]["gene"]["variants"]

    print(
        "gnomAD ATM variants retrieved:",
        len(gnomad_variants)
    )

gnomAD ATM variants retrieved: 13905


In [27]:
gnomad_df = pd.json_normalize(gnomad_variants)

gnomad_df.head()

,variant_id,joint.ac,joint.an
0,11-108227550-A-G,8,1165672
1,11-108227551-T-C,505,1185976
2,11-108227551-TATATAC-T,34,1185980
3,11-108227552-A-C,1,1197114
4,11-108227552-A-G,2,1196996


In [28]:
gnomad_population = gnomad_df.rename(
    columns={
        "joint.ac": "gnomad_ac",
        "joint.an": "gnomad_an"
    }
)

gnomad_population.head()

,variant_id,gnomad_ac,gnomad_an
0,11-108227550-A-G,8,1165672
1,11-108227551-T-C,505,1185976
2,11-108227551-TATATAC-T,34,1185980
3,11-108227552-A-C,1,1197114
4,11-108227552-A-G,2,1196996


In [29]:
matched_variants = candidate_variants.merge(
    gnomad_population,
    on="variant_id",
    how="inner"
)

print("ClinVar candidates:", len(candidate_variants))
print("Matched in gnomAD:", len(matched_variants))

matched_variants.head()

ClinVar candidates: 4534
Matched in gnomAD: 1474


,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,Oncogenicity date last evaluated,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,variant_id,gnomad_ac,gnomad_an
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,NaN,11:108098355,11:108227628,640846,VCV000640846,rs639367,11-108227628-A-G,2,1613596
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,NaN,11:108098357,11:108227630,1337452,VCV001337452,rs1328461,11-108227630-T-G,2,1613628
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,V4L,2025/01/12,NaN,NaN,11:108098361,11:108227634,481224,VCV000481224,rs475312,11-108227634-G-T,2,1613478
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13C,2025/12/20,NaN,NaN,11:108098388,11:108227661,185977,VCV000185977,rs183070,11-108227661-C-T,18,1613346
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13H,2026/01/26,NaN,NaN,11:108098389,11:108227662,188171,VCV000188171,rs186132,11-108227662-G-A,17,1613566


In [30]:
matched_variants["gnomad_af"] = (
    matched_variants["gnomad_ac"]
    / matched_variants["gnomad_an"]
)

matched_variants[
    [
        "Variation",
        "variant_id",
        "gnomad_ac",
        "gnomad_an",
        "gnomad_af"
    ]
].head()

,Variation,variant_id,gnomad_ac,gnomad_an,gnomad_af
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),11-108227628-A-G,2,1613596,0.000001
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),11-108227630-T-G,2,1613628,0.000001
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),11-108227634-G-T,2,1613478,0.000001
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),11-108227661-C-T,18,1613346,0.000011
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),11-108227662-G-A,17,1613566,0.000011


In [31]:
matched_variant_ids = set(
    matched_variants["variant_id"]
)

unmatched_variants = candidate_variants[
    ~candidate_variants["variant_id"].isin(
        matched_variant_ids
    )
].copy()

print(
    "Matched variants:",
    len(matched_variants)
)

print(
    "Unmatched variants:",
    len(unmatched_variants)
)

Matched variants: 1474
Unmatched variants: 3060


In [32]:
matched_variants = (
    matched_variants
    .sort_values(
        by="gnomad_af",
        ascending=False,
        na_position="last"
    )
    .reset_index(drop=True)
)

matched_variants[
    [
        "Variation",
        "Protein change",
        "variant_id",
        "gnomad_ac",
        "gnomad_an",
        "gnomad_af"
    ]
].head(20)

,Variation,Protein change,variant_id,gnomad_ac,gnomad_an,gnomad_af
0,NM_000051.4(ATM):c.3265G>T (p.Ala1089Ser),A1089S,11-108272833-G-T,143,1613882,0.000089
1,NM_000051.4(ATM):c.1440A>C (p.Leu480Phe),L480F,11-108250905-A-C,95,1614080,0.000059
2,NM_000051.4(ATM):c.1837G>T (p.Val613Leu),V613L,11-108252851-G-T,68,1612816,0.000042
3,NM_000051.4(ATM):c.5189G>A (p.Arg1730Gln),R1730Q,11-108301659-G-A,68,1613370,0.000042
4,NM_000051.4(ATM):c.2698A>G (p.Met900Val),M900V,11-108268469-A-G,67,1613962,0.000042
5,NM_000051.4(ATM):c.501G>T (p.Leu167Phe),L167F,11-108243957-G-T,61,1510894,0.000040
6,NM_000051.4(ATM):c.2771G>A (p.Arg924Gln),R924Q,11-108268542-G-A,61,1613960,0.000038
7,NM_000051.4(ATM):c.2689T>A (p.Phe897Ile),F897I,11-108268460-T-A,57,1613938,0.000035
8,NM_000051.4(ATM):c.2418G>T (p.Leu806Phe),L806F,11-108259027-G-T,52,1613726,0.000032
9,NM_000051.4(ATM):c.2603A>G (p.Asp868Gly),D868G,11-108267307-A-G,51,1614000,0.000032


In [33]:
assert matched_variants["variant_id"].notna().all()

assert (
    matched_variants["gnomad_ac"].dropna()
    >= 0
).all()

assert (
    matched_variants["gnomad_an"].dropna()
    >= 0
).all()

assert (
    matched_variants["gnomad_ac"].dropna()
    <= matched_variants["gnomad_an"].dropna()
).all()

print("Validation checks passed!")

Validation checks passed!


In [34]:
processed_folder = project_root / "data" / "processed" 

matched_output_file = (
    processed_folder
    / "atm_clinvar_gnomad_matched.csv"
)

matched_variants.to_csv(
    matched_output_file,
    index=False
)

print(
    "Matched dataset saved to:",
    matched_output_file
)

Matched dataset saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_clinvar_gnomad_matched.csv


In [35]:
unmatched_output_file = (
    processed_folder
    / "atm_clinvar_gnomad_unmatched.csv"
)

unmatched_variants.to_csv(
    unmatched_output_file,
    index=False
)

print(
    "Unmatched dataset saved to:",
    unmatched_output_file
)

Unmatched dataset saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_clinvar_gnomad_unmatched.csv


In [36]:
match_percentage = (
    len(matched_variants)
    / len(candidate_variants)
    * 100
)

print("Final pipeline summary")
print("----------------------")
print("ClinVar candidates:", len(candidate_variants))
print("Matched in gnomAD:", len(matched_variants))
print("Unmatched:", len(unmatched_variants))
print(f"Match percentage: {match_percentage:.2f}%")

Final pipeline summary
----------------------
ClinVar candidates: 4534
Matched in gnomAD: 1474
Unmatched: 3060
Match percentage: 32.51%


In [ ]:
gnomad_url = "https://gnomad.broadinstitute.org/api"

# calling again for the homozygote count and filters
query = """
{
  gene(gene_symbol: "ATM", reference_genome: GRCh38) {
    variants(dataset: gnomad_r4) {
      variant_id
      joint {
        ac
        an
        homozygote_count
        filters
      }
    }
  }
}
"""

In [38]:
response = requests.post(
    gnomad_url,
    json={"query": query},
    timeout=120
)

response.raise_for_status()

gnomad_response = response.json()

if "errors" in gnomad_response:
    raise ValueError(gnomad_response["errors"])

gnomad_variants = (
    gnomad_response["data"]["gene"]["variants"]
)

print(
    "gnomAD ATM variants retrieved:",
    len(gnomad_variants)
)


gnomAD ATM variants retrieved: 13905


In [39]:
gnomad_df = pd.json_normalize(gnomad_variants)

print(gnomad_df.columns.tolist())

gnomad_df.head()

['variant_id', 'joint.ac', 'joint.an', 'joint.homozygote_count', 'joint.filters']


,variant_id,joint.ac,joint.an,joint.homozygote_count,joint.filters
0,11-108227550-A-G,8,1165672,0,[]
1,11-108227551-T-C,505,1185976,3,[]
2,11-108227551-TATATAC-T,34,1185980,0,[]
3,11-108227552-A-C,1,1197114,0,[]
4,11-108227552-A-G,2,1196996,0,[]


In [40]:
gnomad_population = (
    gnomad_df[
        [
            "variant_id",
            "joint.ac",
            "joint.an",
            "joint.homozygote_count",
            "joint.filters"
        ]
    ]
    .rename(
        columns={
            "joint.ac": "gnomad_ac",
            "joint.an": "gnomad_an",
            "joint.homozygote_count": "gnomad_homozygote_count",
            "joint.filters": "gnomad_filters"
        }
    )
    .drop_duplicates(
        subset="variant_id"
    )
    .reset_index(drop=True)
)

gnomad_population.head()

,variant_id,gnomad_ac,gnomad_an,gnomad_homozygote_count,gnomad_filters
0,11-108227550-A-G,8,1165672,0,[]
1,11-108227551-T-C,505,1185976,3,[]
2,11-108227551-TATATAC-T,34,1185980,0,[]
3,11-108227552-A-C,1,1197114,0,[]
4,11-108227552-A-G,2,1196996,0,[]


In [ ]:
# repeated process but includes .. gnomad homozygote
gnomad_population["gnomad_ac"] = pd.to_numeric(
    gnomad_population["gnomad_ac"],
    errors="coerce"
)

gnomad_population["gnomad_an"] = pd.to_numeric(
    gnomad_population["gnomad_an"],
    errors="coerce"
)

gnomad_population["gnomad_homozygote_count"] = pd.to_numeric(
    gnomad_population["gnomad_homozygote_count"],
    errors="coerce"
)

In [42]:
matched_variants = candidate_variants.merge(
    gnomad_population,
    on="variant_id",
    how="inner"
)

print("ClinVar candidates:", len(candidate_variants))
print("Matched in gnomAD:", len(matched_variants))

matched_variants.head()

ClinVar candidates: 4534
Matched in gnomAD: 1474


,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,variant_id,gnomad_ac,gnomad_an,gnomad_homozygote_count,gnomad_filters
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,...,11:108098355,11:108227628,640846,VCV000640846,rs639367,11-108227628-A-G,2,1613596,0,[]
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,...,11:108098357,11:108227630,1337452,VCV001337452,rs1328461,11-108227630-T-G,2,1613628,0,[]
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,V4L,2025/01/12,NaN,...,11:108098361,11:108227634,481224,VCV000481224,rs475312,11-108227634-G-T,2,1613478,0,[]
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13C,2025/12/20,NaN,...,11:108098388,11:108227661,185977,VCV000185977,rs183070,11-108227661-C-T,18,1613346,0,[]
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13H,2026/01/26,NaN,...,11:108098389,11:108227662,188171,VCV000188171,rs186132,11-108227662-G-A,17,1613566,0,[]


In [43]:
matched_variants["gnomad_af"] = (
    matched_variants["gnomad_ac"]
    /
    matched_variants["gnomad_an"].replace(
        0,
        pd.NA
    )
)

In [44]:
matched_variants[
    [
        "Variation",
        "variant_id",
        "gnomad_ac",
        "gnomad_an",
        "gnomad_af",
        "gnomad_homozygote_count",
        "gnomad_filters"
    ]
].head(20)

,Variation,variant_id,gnomad_ac,gnomad_an,gnomad_af,gnomad_homozygote_count,gnomad_filters
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),11-108227628-A-G,2,1613596,1.239468e-06,0,[]
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),11-108227630-T-G,2,1613628,1.239443e-06,0,[]
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),11-108227634-G-T,2,1613478,1.239558e-06,0,[]
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),11-108227661-C-T,18,1613346,1.115694e-05,0,[]
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),11-108227662-G-A,17,1613566,1.053567e-05,0,[]
5,NM_000051.4(ATM):c.41A>G (p.Gln14Arg),11-108227665-A-G,3,1613862,1.858895e-06,0,[]
6,NM_000051.4(ATM):c.50A>G (p.His17Arg),11-108227674-A-G,2,1613864,1.239262e-06,0,[]
7,NM_000051.4(ATM):c.62C>T (p.Thr21Ile),11-108227686-C-T,5,1613644,3.098577e-06,0,[]
8,NM_000051.4(ATM):c.68G>A (p.Arg23Gln),11-108227692-G-A,13,1613478,8.057129e-06,0,[]
9,NM_000051.4(ATM):c.74A>G (p.Lys25Arg),11-108227777-A-G,2,1613592,1.239471e-06,0,[]


In [45]:
output_file = (
    project_root
    / "data"
    / "processed"
    / "atm_clinvar_gnomad_enriched.csv"
)

matched_variants.to_csv(
    output_file,
    index=False
)

print("Saved to:", output_file)

Saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_clinvar_gnomad_enriched.csv
